# CASO PRACTICO

In [0]:
#importando pandas
import pandas as pd

# PASAR LOS CSV EN UN DATA FRAME Y LUEGO UNIRLOS

In [0]:
df_menu1=pd.read_csv("restaurant-menus/restaurant_menus_parte_1.csv")
pd.DataFrame(df_menu1)

In [0]:
df_menu2=pd.read_csv("restaurant-menus/restaurant_menus_parte_2.csv")
pd.DataFrame(df_menu2)

In [0]:
df_menu3=pd.read_csv("restaurant-menus/restaurant_menus_parte_3.csv")
pd.DataFrame(df_menu3)

In [0]:
df_menu4=pd.read_csv("restaurant-menus/restaurant_menus_parte_4.csv")
pd.DataFrame(df_menu4)

In [0]:
df_menu5=pd.read_csv("restaurant-menus/restaurant_menus_parte_5.csv")
pd.DataFrame(df_menu5)

In [0]:
df_menu6=pd.read_csv("restaurant-menus/restaurant_menus_parte_6.csv")
pd.DataFrame(df_menu6)

In [0]:
df_menu7=pd.read_csv("restaurant-menus/restaurant_menus_parte_7.csv")
pd.DataFrame(df_menu7)


In [0]:
df_menu8=pd.read_csv("restaurant-menus/restaurant_menus_parte_8.csv")
pd.DataFrame(df_menu8)

In [0]:
df_menu9=pd.read_csv("restaurant-menus/restaurant_menus_parte_9.csv")
pd.DataFrame(df_menu9)

In [0]:
df_menu10=pd.read_csv("restaurant-menus/restaurant_menus_parte_10.csv")
pd.DataFrame(df_menu10)

In [0]:
#UNIR TODOS LOS DF_MENUS EN UNO SOLO DF_MENU
df_menu = pd.concat([df_menu1, df_menu2, df_menu3, df_menu4, df_menu5, df_menu6, df_menu7, df_menu8, df_menu9, df_menu10], ignore_index=True)
pd.DataFrame(df_menu)


In [0]:
#COLOCAR EN UN DATAFRAME RESTAURANTS
df_restaurantes = pd.read_csv("restaurants/restaurants.csv")
pd.DataFrame(df_restaurantes)

# LIMPIEZA DE LOS DATAFRAME

In [0]:
#DIMENSIONES DEL DF_MENU
df_menu.shape


In [0]:
#DIMENSIONES DEL DF_RESTAURANTES
df_restaurantes.shape

In [0]:
#INFORMACION DE DATOS QUE CONTIENE LOS DATAFRAMES
df_menu.info()
df_restaurantes.info()


In [0]:
#CANTIDAD DE VALORES NULOS DE CADA DATAFRAME
df_menu.isnull().sum()
#df_restaurantes.isnull().sum()


In [0]:
#CANTIDAD DE VALORES NULOS DE CADA DATAFRAME
df_restaurantes.isnull().sum()

# PARTE 2 - Transform

## DF_RESTAURANTES

In [0]:
df_restaurantes

In [0]:
df_restaurantes["price_range_modificado"]=df_restaurantes["price_range"].replace
({
            "$":"Economico"
          ,"$$":"Moderadamente Caro"
          ,"$$$":"Caro"
          ,"$$$$":"Muy Caro"
          #,"np.nan":"Sin Rango"
})
df_restaurantes

In [0]:
##.str.split(',', expand=True) para crear tres nuevas columnas: calle, ciudad y 
#estado_zip. Asegúrese de aplicar .str.strip() a la nueva columna ciudad para 
#quitar espacios ocultos. df_restaurantes
df_restaurantes["calle"]=df_restaurantes["full_address"].str.split(',', expand=True)[0]
df_restaurantes["ciudad"]=df_restaurantes["full_address"].str.split(',', expand=True)[1].str.strip()
df_restaurantes["estado_zip"]=df_restaurantes["full_address"].str.split(',', expand=True)[3]
df_restaurantes.head(10)

In [0]:
#Elimine los restaurantes que no tengan puntaje (score nulo) o 
#cuyo puntaje sea 0. Estandarice la columna category a minúsculas.
df_restaurantes=df_restaurantes.dropna(subset=["score"])
df_restaurantes=df_restaurantes[df_restaurantes["score"]!=0]
df_restaurantes["category"]=df_restaurantes["category"].str.lower()
df_restaurantes

In [0]:
df_menu

In [0]:
#La columna price viene como texto (ej. "15,99 USD"). Cree una función lambda o 
#use .str.replace() para eliminar el texto " USD", reemplazar la coma por un #punto, y castear el resultado a formato decimal (float). 
df_menu["price"]=df_menu["price"].astype(str).str.replace("USD","").str.replace(",",".").astype(float)
df_menu.head(10)

In [0]:
#Elimine los ítems del menú con precio igual a 0.0 o nulos. 
df_menu=df_menu.dropna(subset=["price"])
df_menu=df_menu[df_menu["price"]!=0.0]
df_menu.head(10)

## Control de Memoria RAM

In [0]:
#Para no colapsar la memoria del servidor, el CEO solo quiere analizar los restaurantes 
#consolidados. Filtre df_restaurantes para quedarse únicamente con aquellos que 
#tengan más de 100 calificaciones (ratings). 
df_restaurantes=df_restaurantes[df_restaurantes["ratings"]>100]
df_restaurantes.head(10)


In [0]:
#Realice un INNER JOIN (pd.merge) entre su df_restaurantes filtrado y df_menus 
#utilizando el ID del restaurante. Guarde este cruce en un DataFrame llamado df_master. 

df_master=pd.merge(df_restaurantes,df_menu,how="inner",left_on="id",right_on="restaurant_id")
df_master.head(10)


# Parte 3 - Load

In [0]:
#Instalando sqlalchemy
%pip install sqlalchemy

In [0]:
#Importe la librería sqlalchemy y genere un motor (create_engine) para una base de 
#datos SQLite local llamada delivery_insights.db. 
from sqlalchemy import create_engine
engine = create_engine('sqlite:///delivery_insights.db')

In [0]:
#Utilice df_master.to_sql() para cargar su tabla final a la base de datos con el nombre 
#master_food_data. Use if_exists="replace" y index=False. 
df_master.to_sql("master_food_data", engine, if_exists="replace", index=False)

# Parte 4 - BI

In [0]:
#Conéctese a su base de datos SQLite mediante pd.read_sql(text(query), conn). 
con = engine.connect()
#Penetración Geográfica: ¿Cuáles son las 5 ciudades (ciudad) con la mayor cantidad de restaurantes consolidados
q1 = """
SELECT ciudad, count(ciudad) as cantidad
FROM master_food_data
WHERE ciudad IS NOT NULL
GROUP BY ciudad
ORDER BY cantidad DESC
LIMIT 5
"""
top5_ciudades = pd.read_sql(q1, con)
print(top5_ciudades.head(5).to_string(index=False))
con.close()



In [0]:
#Análisis de Oportunidad de Precio: El CEO cree que todos los restaurantes en el mercado son caros. Usando los datos, #muestre la distribución de la columna rango_de_precios a nivel general (¿Cuántos hay en "Económico", "Caro", etc.?).
con = engine.connect()

q2 = """
SELECT price_range_modificado, count(restaurant_id) as cantidad
FROM master_food_data
WHERE price_range_modificado IS NOT NULL
GROUP BY price_range_modificado
ORDER BY cantidad DESC

"""
rango_precio = pd.read_sql(q2, con)
print(rango_precio.to_string(index=False))
con.close()

In [0]:
#Para las 3 ciudades más populares obtenidas en la pregunta 1, calcule el precio promedio de los platos en el menú. ¿En 
#qué ciudad es más caro comer en promedio?
con = engine.connect()

q3 = """
SELECT ciudad, round(AVG(price),2) as promedio
FROM master_food_data
WHERE ciudad IN ('Milwaukee', 'Seattle', 'Lynnwood')
GROUP BY ciudad
ORDER BY promedio DESC
"""
promedio_precio = pd.read_sql(q3, con)
print(promedio_precio.to_string(index=False))
con.close()

In [0]:
#¿Un restaurante más caro garantiza una mejor experiencia para el usuario? Escriba un query que calcule el puntaje (score) promedio agrupado por rango_de_precios.
con = engine.connect()

q4 = """
SELECT price_range_modificado, round(AVG(score),2) as puntaje
FROM master_food_data
WHERE price_range_modificado IS NOT NULL
GROUP BY price_range_modificado
ORDER BY puntaje DESC
"""
promedio_puntaje = pd.read_sql(q4, con)
print(promedio_puntaje.to_string(index=False))
con.close()

In [0]:
#IMPORTANDO LIBRERIAS PARA LAS GRAFICAS
!pip install seaborn 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [0]:
barras = plt.barh(top5_ciudades['ciudad'],top5_ciudades['cantidad'],label='Valores')
plt.title("Grafico de barras")
plt.legend()
# Etiquetas encima de cada barra
plt.bar_label(barras)
plt.xlabel("Cantidad")
plt.ylabel("Ciudad")
plt.show()

In [0]:
#plt.pie(rango_precio["cantidad"], labels = rango_precio["price_range_modificado"])
separacion = (0.1,) + (0,) * (len(rango_precio['cantidad']) - 1) # separa solo la primera rebanada
plt.pie(rango_precio["cantidad"], labels = rango_precio["price_range_modificado"],
        explode = separacion, # 0 = unido, >0 = separado
        autopct='%1.1f%%', # formato del porcentaje: que muestre un decimal
        shadow = True, # sombra 3D
        startangle=90, # angulo de inicio, es para girar el grafico
        )
plt.show()

In [0]:
barras = plt.bar(promedio_precio['ciudad'],promedio_precio['promedio']
                 ,label='Valores'
                 ,color = 'green'
                 ,edgecolor = 'navy'     
                 )
plt.title("Grafico de barras")
plt.legend()
# Etiquetas encima de cada barra
plt.bar_label(barras)
plt.xlabel("Ciudad")
plt.ylabel("Promedio")
plt.show()


In [0]:
plt.plot(promedio_puntaje['price_range_modificado'],promedio_puntaje['puntaje']
         ,color = 'purple'
         ,linestyle = '--', # solida (-) , discontinua (,y,
         linewidth = 2, # grosor de la linea
         marker = 'o', # Punto (o), cuadrado (s), triangulo (^)
         markersize = 10, # tamaño del marcador
         label = 'Precio-Calidad'
         )
plt.title('Comparacion de Precio-Calidad')
plt.xlabel('Rango')
plt.ylabel('Puntaje')
plt.grid(True)
plt.legend()
plt.show()